# AML Case Investigation & SAR Decision Analytics


In [ ]:
import pandas as pd
cases=pd.read_csv('../data/processed/investigation_cases.csv')
sar=pd.read_csv('../data/processed/sar_decisions.csv')
cases.head()


## Case disposition analysis

In [ ]:
cases['disposition'].value_counts()


## SAR conversion by case type

In [ ]:
cases.groupby('case_type')['sar_flag'].mean().sort_values(ascending=False)


## High-priority investigation queue

In [ ]:
cases[cases['priority'].isin(['High','Critical'])].sort_values('case_risk_score',ascending=False).head(20)


## EDA + Feature Engineering

This section adds a simple, practical EDA and feature-engineering workflow to the analysis already in this notebook. The goal is to demonstrate data-quality review, exploratory analysis, and creation of useful analytical features without making the project unnecessarily advanced.

**Workflow:** Load/confirm data → dataset review → missing values → duplicates → datatype/column checks → data quality → outliers/ranges → KPI checks → feature engineering → business-rule checks → summary statistics → insights.


In [ ]:
# EDA: use the dataframe already created in this notebook
# This keeps the existing project logic unchanged.
_df_candidates = ['df', 'data', 'alerts', 'transactions', 'cases', 'customers', 'rules']
eda_df = None

for _name in _df_candidates:
    if _name in globals() and hasattr(globals()[_name], 'columns'):
        eda_df = globals()[_name].copy()
        print(f"EDA dataframe: {_name}")
        break

if eda_df is None:
    print("No existing dataframe was detected. Run the earlier data-loading cells first.")
else:
    print("Shape:", eda_df.shape)
    display(eda_df.head())
    print("\nData types:")
    display(eda_df.dtypes.to_frame("dtype"))


In [ ]:
# Data quality checks
if eda_df is not None:
    quality_summary = {
        "rows": len(eda_df),
        "columns": eda_df.shape[1],
        "missing_values": int(eda_df.isna().sum().sum()),
        "duplicate_rows": int(eda_df.duplicated().sum())
    }
    display(pd.DataFrame([quality_summary]))

    missing = eda_df.isna().sum().sort_values(ascending=False)
    missing = missing[missing > 0]
    print("Columns with missing values:")
    display(missing.to_frame("missing_count") if len(missing) else pd.DataFrame({"status":["No missing values found"]}))

    # Standardize column names on the EDA copy only.
    eda_df.columns = (
        eda_df.columns.astype(str).str.strip().str.lower()
        .str.replace(r'[^a-z0-9]+', '_', regex=True).str.strip('_')
    )
    print("Standardized columns:", list(eda_df.columns))


In [ ]:
# Simple numeric review and outlier flags (IQR method)
if eda_df is not None:
    numeric_cols = eda_df.select_dtypes(include='number').columns.tolist()
    if numeric_cols:
        display(eda_df[numeric_cols].describe().T)

        outlier_summary = []
        for col in numeric_cols:
            q1 = eda_df[col].quantile(0.25)
            q3 = eda_df[col].quantile(0.75)
            iqr = q3 - q1
            if pd.notna(iqr) and iqr > 0:
                lower, upper = q1 - 1.5*iqr, q3 + 1.5*iqr
                count = int(((eda_df[col] < lower) | (eda_df[col] > upper)).sum())
                outlier_summary.append([col, count])
        display(pd.DataFrame(outlier_summary, columns=['field','potential_outliers']))
    else:
        print("No numeric columns available for numeric/outlier review.")

# In AML/fraud work, unusual values can be meaningful.
# We flag them for investigation rather than automatically deleting them.


In [ ]:
# Practical feature engineering based on columns that already exist
if eda_df is not None:
    created_features = []

    # Amount-based feature
    amount_col = next((c for c in eda_df.columns if c in ['amount','transaction_amount','txn_amount']), None)
    if amount_col:
        median_amt = eda_df[amount_col].median()
        eda_df['above_median_amount_flag'] = (eda_df[amount_col] > median_amt).astype(int)
        created_features.append('above_median_amount_flag')

    # Risk-score feature
    risk_col = next((c for c in eda_df.columns if 'risk_score' in c), None)
    if risk_col:
        eda_df['high_risk_flag'] = (eda_df[risk_col] >= eda_df[risk_col].quantile(0.75)).astype(int)
        created_features.append('high_risk_flag')

    # Date feature
    date_col = next((c for c in eda_df.columns if 'date' in c or 'timestamp' in c), None)
    if date_col:
        _dates = pd.to_datetime(eda_df[date_col], errors='coerce')
        eda_df['activity_month'] = _dates.dt.to_period('M').astype(str)
        created_features.append('activity_month')

    print("Features created:", created_features if created_features else
          "No generic feature was forced; keep the project's existing domain-specific features.")
    display(eda_df.head())


In [ ]:
# Final validation and analysis-ready export
if eda_df is not None:
    print("Final EDA dataset shape:", eda_df.shape)
    print("Duplicate rows:", int(eda_df.duplicated().sum()))
    print("Total missing values:", int(eda_df.isna().sum().sum()))

    # Export is optional; uncomment when you want a clean analysis-ready file.
    # eda_df.to_csv("data/processed/analysis_ready_dataset.csv", index=False)

    print("\nEDA + Feature Engineering complete.")
